## 1. Opções dos EUA

In [20]:
import yfinance as yf 

In [26]:
 # Importa a biblioteca yfinance para acessar dados financeiros

##### Defina o ticker
symbol = 'AAPL'  # Define o símbolo da ação (Apple)
stock = yf.Ticker(symbol)  # Cria um objeto Ticker para o símbolo especificado

##### Coleta de dados das opções - datas de vencimento
options = stock.options  # Obtém as datas de vencimento das opções disponíveis para o ticker

In [27]:
##### A option chain
for option_symbol in options:
    option_chain = stock.option_chain(option_symbol)  # Obtém a cadeia de opções para uma data de vencimento específica
    
    # Opções de compra (calls)
    call_options = option_chain.calls
    print(f"Call Options para {option_symbol}:")
    print(call_options)

    # Opções de venda (puts)
    put_options = option_chain.puts
    print(f"Put Options para {option_symbol}:")
    print(put_options)

## 2. Opções do Brasil (B3)

In [14]:
import pandas as pd  # Importa a biblioteca pandas para manipulação de dados
import requests  # Importa a biblioteca requests para fazer requisições HTTP

subjacente = 'VALE3'  # Define o ativo subjacente (Vale S.A.)

##### Para um vencimento
vencimento = '2024-06-21'  # Define uma data de vencimento específica (formato YYYY-MM-DD)

In [16]:
def optionchaindate(subjacente, vencimento):
    # URL para obter dados das opções de um ativo específico e uma data de vencimento específica
    url = f'https://opcoes.net.br/listaopcoes/completa?idAcao={subjacente}&listarVencimentos=false&cotacoes=true&vencimentos={vencimento}'
    r = requests.get(url).json()  # Faz a requisição HTTP e obtém a resposta em formato JSON
    # Extrai e organiza os dados de opções em um DataFrame do pandas
    x = ([subjacente, vencimento, i[0].split('_')[0], i[2], i[3], i[5], i[8], i[9], i[10]] for i in r['data']['cotacoesOpcoes'])
    return pd.DataFrame(x, columns=['subjacente', 'vencimento', 'ativo', 'tipo', 'modelo', 'strike', 'preco', 'negocios', 'volume'])

# Chama a função para obter os dados de opções para o subjacente e vencimento especificados
optionchaindate(subjacente, vencimento)

,subjacente,vencimento,ativo,tipo,modelo,strike,preco,negocios,volume
0,VALE3,2024-06-21,VALEF11,CALL,A,97.99,0.01,1.0,1.0
1,VALE3,2024-06-21,VALEF12,CALL,A,107.99,0.01,2.0,13.0
2,VALE3,2024-06-21,VALEF13,CALL,A,117.99,0.01,1.0,12.0
3,VALE3,2024-06-21,VALEF14,CALL,A,131.99,0.01,1.0,1000.0
4,VALE3,2024-06-21,VALEF70,CALL,E,63.49,0.01,16.0,621.0
...,...,...,...,...,...,...,...,...,...
213,VALE3,2024-06-21,VALER900,PUT,E,77.99,17.17,11.0,18782.0
214,VALE3,2024-06-21,VALER909,PUT,E,90.99,29.94,1.0,11976.0
215,VALE3,2024-06-21,VALER922,PUT,E,89.49,28.29,9.0,621796.0
216,VALE3,2024-06-21,VALER950,PUT,E,82.99,22.07,1.0,13242.0


In [18]:
##### Para todos os vencimentos
def optionchain(subjacente):
    # URL para obter dados das opções de todos os vencimentos disponíveis para um ativo específico
    url2 = f'https://opcoes.net.br/listaopcoes/completa?idLista=ML&idAcao={subjacente}&listarVencimentos=true&cotacoes=true'
    r = requests.get(url2).json()  # Faz a requisição HTTP e obtém a resposta em formato JSON
    vencimentos = [i['value'] for i in r['data']['vencimentos']]  # Extrai todas as datas de vencimento disponíveis
    # Concatena os dados de opções de todas as datas de vencimento em um único DataFrame
    df = pd.concat([optionchaindate(subjacente, vencimento) for vencimento in vencimentos])
    return df

# Chama a função para obter os dados de opções para todos os vencimentos do subjacente especificado
optionchain(subjacente)

,subjacente,vencimento,ativo,tipo,modelo,strike,preco,negocios,volume
0,VALE3,2024-10-18,VALEJ83,CALL,A,78.55,0.01,1.0,3.0
1,VALE3,2024-10-18,VALEJ118,CALL,E,118.55,NaN,NaN,NaN
2,VALE3,2024-10-18,VALEJ119,CALL,A,119.55,NaN,NaN,NaN
3,VALE3,2024-10-18,VALEJ121,CALL,A,121.55,NaN,NaN,NaN
4,VALE3,2024-10-18,VALEJ122,CALL,E,122.55,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
13,VALE3,2026-06-19,VALER702,PUT,E,70.22,NaN,NaN,NaN
14,VALE3,2026-06-19,VALER752,PUT,E,75.22,7.50,1.0,750.0
15,VALE3,2026-06-19,VALER897,PUT,E,89.72,19.13,1.0,1913.0
0,VALE3,2026-08-21,VALEH760,CALL,E,76.00,7.15,2.0,1426.0


In [19]:
# Filtrando um pouco
chain = optionchaindate(subjacente, vencimento)  # Obtém os dados de opções para um vencimento específico
chain_filtered = chain[chain['negocios'] >= 10]  # Filtra as opções que tiveram pelo menos 10 negócios

# Separando calls e puts
calls = chain_filtered[chain_filtered['tipo'] == 'CALL']  # Separa as opções de compra (CALL)
puts = chain_filtered[chain_filtered['tipo'] == 'PUT']  # Separa as opções de venda (PUT)

# Merge de calls e puts de acordo com a coluna 'strike'
merged_df = pd.merge(calls[['strike', 'ativo']], puts[['strike', 'ativo']], on='strike', suffixes=('_call', '_put'))

# Renomeando colunas
merged_df.rename(columns={'ativo_call': 'ativo_call', 'ativo_put': 'ativo_put'}, inplace=True)
merged_df['subjacente'] = subjacente  # Adiciona a coluna 'subjacente' para identificar o ativo subjacente
pcpairs = merged_df[['subjacente', 'ativo_call', 'ativo_put', 'strike']]  # Seleciona as colunas desejadas para o resultado final

# Converte os pares de calls e puts em uma lista e conta o número de pares
pclist = pcpairs.values.tolist()
len(pclist)

11